In [ ]:
from google.colab import files
uploaded = files.upload()


Saving credits.csv to credits.csv


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving movies.csv to movies.csv


In [ ]:
import pandas as pd

movies = pd.read_csv('movies.csv')
credits = pd.read_csv('credits.csv')

In [ ]:
movies = movies.merge(credits, on='title')

In [ ]:
print(movies.columns)

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'movie_id', 'cast', 'crew', 'movie_id_y', 'cast_y',
       'crew_y', 'movie_id_y', 'cast_y', 'crew_y'],
      dtype='object')


In [ ]:
movies.rename(columns={
    'movie_id_x': 'movie_id',
    'cast_x': 'cast',
    'crew_x': 'crew'
}, inplace=True)

In [ ]:
print(movies.columns)

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'movie_id', 'cast', 'crew', 'movie_id_y', 'cast_y',
       'crew_y', 'movie_id_y', 'cast_y', 'crew_y'],
      dtype='object')


In [ ]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [ ]:
movies.dropna(inplace=True)

In [ ]:
import ast

def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert)

In [ ]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=1000, stop_words='english')
vectors = cv.fit_transform(movies['tags']).toarray()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

In [ ]:
def recommend(movie):
    index = movies[movies['title'] == movie].index[0]
    distances = similarity[index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    return [movies.iloc[i[0]].title for i in movies_list]

In [ ]:
recommend("Avatar")

['Star Trek Into Darkness',
 'The Dark Knight',
 'Aliens',
 'Armageddon',
 'Pearl Harbor']

In [ ]:
import pickle

pickle.dump(movies, open('movies.pkl','wb'))
pickle.dump(similarity, open('similarity.pkl','wb'))

In [ ]:
import pickle

# Save
pickle.dump(movies, open('movies.pkl','wb'))
pickle.dump(similarity, open('similarity.pkl','wb'))

# Zip (important)
import zipfile

with zipfile.ZipFile('model.zip', 'w') as zipf:
    zipf.write('movies.pkl')
    zipf.write('similarity.pkl')

In [ ]:
from google.colab import files
files.download('model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>